# Visualizacion EURUSD: Dinamica Diaria vs Horaria

Analisis visual del par EUR/USD en ambas temporalidades:
- Precio (OHLC / candlestick)
- Volumen
- Volatilidad (ATR)
- Medias moviles (50, 150, 200 SMA)
- Trend Template (Etapa 2 Minervini)

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from stages.trend_template import evaluate_trend_template

pd.set_option("display.float_format", "{:.5f}".format)
plt.rcParams["figure.figsize"] = (18, 6)
plt.rcParams["figure.dpi"] = 100
print("Imports OK")

## 1. Cargar datos

In [ ]:
daily = pd.read_csv(
    project_root / "data" / "monedas" / "EURUSD.csv",
    parse_dates=["date"], index_col="date",
)
hourly = pd.read_csv(
    project_root / "data" / "monedas_hora" / "EURUSD.csv",
    parse_dates=["date"], index_col="date",
)

print(f"EURUSD diario:  {len(daily):,} barras, {daily.index.min().date()} a {daily.index.max().date()}")
print(f"EURUSD horario: {len(hourly):,} barras, {hourly.index.min()} a {hourly.index.max()}")
print()
display(daily.describe())
print()
display(hourly.describe())

## 2. Precio y Medias Moviles

In [ ]:
def add_moving_averages(df, periods=[50, 150, 200]):
    result = df.copy()
    for p in periods:
        result[f"sma_{p}"] = result["close"].rolling(p).mean()
    return result

def add_atr(df, period=14):
    result = df.copy()
    high_low = result["high"] - result["low"]
    high_close = (result["high"] - result["close"].shift(1)).abs()
    low_close = (result["low"] - result["close"].shift(1)).abs()
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    result["atr"] = tr.rolling(period).mean()
    result["atr_pct"] = result["atr"] / result["close"] * 100
    return result

daily_ma = add_moving_averages(add_atr(daily))
hourly_ma = add_moving_averages(add_atr(hourly))

print("Medias moviles y ATR calculados.")

In [ ]:
def plot_price_with_sma(df, title, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(18, 7))
    ax.plot(df.index, df["close"], label="Close", color="#2c3e50", linewidth=0.8, alpha=0.9)
    colors = {"sma_50": "#3498db", "sma_150": "#e67e22", "sma_200": "#e74c3c"}
    for col, color in colors.items():
        if col in df.columns:
            ax.plot(df.index, df[col], label=col.upper(), color=color, linewidth=1, alpha=0.7)
    ax.set_title(title, fontweight="bold", fontsize=14)
    ax.legend(loc="upper left")
    ax.grid(True, alpha=0.3)
    ax.set_ylabel("Precio")
    return ax

fig, axes = plt.subplots(2, 1, figsize=(18, 12))
plot_price_with_sma(daily_ma, "EURUSD Diario - Precio y SMAs", ax=axes[0])
plot_price_with_sma(hourly_ma, "EURUSD Horario - Precio y SMAs", ax=axes[1])
plt.tight_layout()
plt.show()

## 3. Volumen

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(18, 10))

for ax, df, title in [
    (axes[0], daily, "EURUSD Diario - Volumen"),
    (axes[1], hourly, "EURUSD Horario - Volumen"),
]:
    colors = np.where(df["close"] >= df["open"], "#27ae60", "#e74c3c")
    ax.bar(df.index, df["volume"], color=colors, alpha=0.6, width=1 if "Diario" in title else 0.04)
    vol_sma = df["volume"].rolling(50).mean()
    ax.plot(df.index, vol_sma, color="#3498db", linewidth=1.2, label="SMA(50) Vol")
    ax.set_title(title, fontweight="bold", fontsize=14)
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylabel("Volumen")

plt.tight_layout()
plt.show()

## 4. Volatilidad (ATR)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 10))

for col_idx, (df, label) in enumerate([(daily_ma, "Diario"), (hourly_ma, "Horario")]):
    # ATR absoluto
    axes[0, col_idx].plot(df.index, df["atr"], color="#8e44ad", linewidth=0.8)
    axes[0, col_idx].set_title(f"EURUSD {label} - ATR(14)", fontweight="bold")
    axes[0, col_idx].set_ylabel("ATR")
    axes[0, col_idx].grid(True, alpha=0.3)
    
    # ATR como % del precio
    axes[1, col_idx].plot(df.index, df["atr_pct"], color="#e67e22", linewidth=0.8)
    axes[1, col_idx].set_title(f"EURUSD {label} - ATR(14) como % del precio", fontweight="bold")
    axes[1, col_idx].set_ylabel("ATR %")
    axes[1, col_idx].grid(True, alpha=0.3)
    axes[1, col_idx].axhline(y=df["atr_pct"].median(), color="gray", linestyle="--", alpha=0.5, label=f"Mediana: {df['atr_pct'].median():.3f}%")
    axes[1, col_idx].legend()

plt.tight_layout()
plt.show()

## 5. Retornos: distribucion y autocorrelacion

In [ ]:
daily_returns = daily["close"].pct_change().dropna()
hourly_returns = hourly["close"].pct_change().dropna()

fig, axes = plt.subplots(2, 2, figsize=(18, 10))

# Histogramas
for ax, rets, label in [
    (axes[0, 0], daily_returns, "Diario"),
    (axes[0, 1], hourly_returns, "Horario"),
]:
    ax.hist(rets, bins=100, color="#3498db", alpha=0.7, edgecolor="white")
    ax.axvline(rets.mean(), color="red", linestyle="--", label=f"Media: {rets.mean():.5f}")
    ax.axvline(0, color="black", linewidth=0.5)
    ax.set_title(f"EURUSD {label} - Distribucion de Retornos", fontweight="bold")
    ax.set_xlabel("Retorno")
    ax.legend()
    ax.grid(True, alpha=0.3)

# Retornos acumulados
for ax, rets, label in [
    (axes[1, 0], daily_returns, "Diario"),
    (axes[1, 1], hourly_returns, "Horario"),
]:
    cumret = (1 + rets).cumprod() - 1
    ax.plot(cumret.index, cumret * 100, color="#2c3e50", linewidth=0.8)
    ax.fill_between(cumret.index, cumret * 100, 0, alpha=0.1, color="#3498db")
    ax.set_title(f"EURUSD {label} - Retorno Acumulado", fontweight="bold")
    ax.set_ylabel("Retorno Acum. (%)")
    ax.axhline(0, color="black", linewidth=0.5)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Retornos diarios:  media={daily_returns.mean():.6f}, std={daily_returns.std():.6f}, skew={daily_returns.skew():.3f}, kurtosis={daily_returns.kurtosis():.3f}")
print(f"Retornos horarios: media={hourly_returns.mean():.6f}, std={hourly_returns.std():.6f}, skew={hourly_returns.skew():.3f}, kurtosis={hourly_returns.kurtosis():.3f}")

## 6. Drawdown historico

In [ ]:
def compute_drawdown(close):
    peak = close.cummax()
    dd = (close - peak) / peak
    return dd

fig, axes = plt.subplots(2, 1, figsize=(18, 10))

for ax, df, label in [
    (axes[0], daily, "Diario"),
    (axes[1], hourly, "Horario"),
]:
    dd = compute_drawdown(df["close"])
    ax.fill_between(dd.index, dd * 100, 0, color="#e74c3c", alpha=0.4)
    ax.plot(dd.index, dd * 100, color="#c0392b", linewidth=0.5)
    ax.set_title(f"EURUSD {label} - Drawdown", fontweight="bold", fontsize=14)
    ax.set_ylabel("Drawdown (%)")
    ax.grid(True, alpha=0.3)
    max_dd = dd.min()
    max_dd_date = dd.idxmin()
    ax.annotate(
        f"Max DD: {max_dd:.1%} ({max_dd_date.strftime('%Y-%m-%d')})",
        xy=(max_dd_date, max_dd * 100),
        xytext=(30, -20), textcoords="offset points",
        arrowprops=dict(arrowstyle="->", color="black"),
        fontsize=10, fontweight="bold",
    )

plt.tight_layout()
plt.show()

## 7. Trend Template (Etapa 2 Minervini)

In [ ]:
daily_template = evaluate_trend_template(daily)
hourly_template = evaluate_trend_template(hourly)

daily_pct = daily_template["trend_template"].sum() / len(daily_template)
hourly_pct = hourly_template["trend_template"].sum() / len(hourly_template)

print(f"Diario:  {daily_template['trend_template'].sum():,} / {len(daily_template):,} barras en Etapa 2 ({daily_pct:.1%})")
print(f"Horario: {hourly_template['trend_template'].sum():,} / {len(hourly_template):,} barras en Etapa 2 ({hourly_pct:.1%})")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(18, 12))

for ax, ohlc, template, label in [
    (axes[0], daily, daily_template, "Diario"),
    (axes[1], hourly, hourly_template, "Horario"),
]:
    ax.plot(ohlc.index, ohlc["close"], color="#2c3e50", linewidth=0.8, label="Close")
    
    stage2 = template[template["trend_template"] == True]
    if len(stage2) > 0:
        stage2_close = ohlc.loc[stage2.index, "close"]
        groups = (stage2.index.to_series().diff() > pd.Timedelta(days=2 if label == "Diario" else hours=2)).cumsum()
        for _, group_idx in stage2.groupby(groups).groups.items():
            idx = stage2.index[stage2.index.isin(group_idx)]
            if len(idx) > 0:
                ax.axvspan(idx[0], idx[-1], color="#27ae60", alpha=0.15)
    
    ax.set_title(f"EURUSD {label} - Zonas en Etapa 2 (verde)", fontweight="bold", fontsize=14)
    ax.set_ylabel("Precio")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Comparacion de rangos por año

In [ ]:
daily_stats = daily.copy()
daily_stats["year"] = daily_stats.index.year
daily_stats["daily_range"] = (daily_stats["high"] - daily_stats["low"]) / daily_stats["close"] * 100

yearly = daily_stats.groupby("year").agg(
    open=("open", "first"),
    close=("close", "last"),
    high=("high", "max"),
    low=("low", "min"),
    avg_daily_range=("daily_range", "mean"),
    avg_volume=("volume", "mean"),
    n_bars=("close", "count"),
)
yearly["annual_return"] = (yearly["close"] / yearly["open"] - 1) * 100
yearly["annual_range"] = (yearly["high"] - yearly["low"]) / yearly["low"] * 100

display(yearly.style.format({
    "open": "{:.4f}",
    "close": "{:.4f}",
    "high": "{:.4f}",
    "low": "{:.4f}",
    "avg_daily_range": "{:.3f}%",
    "avg_volume": "{:,.0f}",
    "n_bars": "{:.0f}",
    "annual_return": "{:+.2f}%",
    "annual_range": "{:.2f}%",
}).background_gradient(subset=["annual_return"], cmap="RdYlGn", vmin=-10, vmax=10))

## 9. Volatilidad por hora del dia (solo horario)

In [ ]:
hourly_stats = hourly.copy()
hourly_stats["hour"] = hourly_stats.index.hour
hourly_stats["bar_range"] = (hourly_stats["high"] - hourly_stats["low"]) / hourly_stats["close"] * 10000  # en pips
hourly_stats["abs_return"] = hourly_stats["close"].pct_change().abs() * 10000  # en pips

by_hour = hourly_stats.groupby("hour").agg(
    avg_range_pips=("bar_range", "mean"),
    avg_abs_return_pips=("abs_return", "mean"),
    avg_volume=("volume", "mean"),
    n_bars=("close", "count"),
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].bar(by_hour.index, by_hour["avg_range_pips"], color="#3498db", alpha=0.7)
axes[0].set_title("Rango promedio por hora (pips)", fontweight="bold")
axes[0].set_xlabel("Hora (UTC)")
axes[0].set_ylabel("Pips")
axes[0].grid(True, alpha=0.3)

axes[1].bar(by_hour.index, by_hour["avg_volume"], color="#27ae60", alpha=0.7)
axes[1].set_title("Volumen promedio por hora", fontweight="bold")
axes[1].set_xlabel("Hora (UTC)")
axes[1].set_ylabel("Volumen")
axes[1].grid(True, alpha=0.3)

axes[2].bar(by_hour.index, by_hour["avg_abs_return_pips"], color="#e67e22", alpha=0.7)
axes[2].set_title("Retorno absoluto promedio por hora (pips)", fontweight="bold")
axes[2].set_xlabel("Hora (UTC)")
axes[2].set_ylabel("Pips")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nEstadisticas por hora:")
display(by_hour.style.format({
    "avg_range_pips": "{:.2f}",
    "avg_abs_return_pips": "{:.2f}",
    "avg_volume": "{:,.0f}",
    "n_bars": "{:,.0f}",
}))

## 10. Resumen comparativo

In [ ]:
summary = pd.DataFrame({
    "Metrica": [
        "Barras totales",
        "Periodo",
        "Precio min",
        "Precio max",
        "Precio actual",
        "Retorno total",
        "Max Drawdown",
        "Volatilidad diaria (std)",
        "ATR(14) promedio",
        "ATR(14) % promedio",
        "Volumen promedio",
        "% tiempo en Etapa 2",
    ],
    "Diario": [
        f"{len(daily):,}",
        f"{daily.index.min().date()} a {daily.index.max().date()}",
        f"{daily['low'].min():.5f}",
        f"{daily['high'].max():.5f}",
        f"{daily['close'].iloc[-1]:.5f}",
        f"{(daily['close'].iloc[-1] / daily['close'].iloc[0] - 1):.2%}",
        f"{compute_drawdown(daily['close']).min():.2%}",
        f"{daily_returns.std():.6f}",
        f"{daily_ma['atr'].mean():.5f}",
        f"{daily_ma['atr_pct'].mean():.3f}%",
        f"{daily['volume'].mean():,.0f}",
        f"{daily_pct:.1%}",
    ],
    "Horario": [
        f"{len(hourly):,}",
        f"{hourly.index.min()} a {hourly.index.max()}",
        f"{hourly['low'].min():.5f}",
        f"{hourly['high'].max():.5f}",
        f"{hourly['close'].iloc[-1]:.5f}",
        f"{(hourly['close'].iloc[-1] / hourly['close'].iloc[0] - 1):.2%}",
        f"{compute_drawdown(hourly['close']).min():.2%}",
        f"{hourly_returns.std():.6f}",
        f"{hourly_ma['atr'].mean():.5f}",
        f"{hourly_ma['atr_pct'].mean():.3f}%",
        f"{hourly['volume'].mean():,.0f}",
        f"{hourly_pct:.1%}",
    ],
})

display(summary.set_index("Metrica").style.set_properties(**{"text-align": "right"}))

print("\nNota: el ATR horario es naturalmente menor al diario (~1/sqrt(24) aprox).")
print("El trend template en horario tiene mas granularidad y puede captar transiciones mas finas.")